In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
icustays_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "icustays_clean.parquet",
    columns=["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
)

icustays_df["intime"] = pd.to_datetime(icustays_df["intime"])
icustays_df["outtime"] = pd.to_datetime(icustays_df["outtime"])

print(icustays_df.shape)
print(icustays_df["stay_id"].duplicated().sum())

(94444, 5)
0


In [3]:
chart_item_map = {
    220045: "heart_rate",
    220210: "resp_rate",
    220277: "spo2",
    220179: "sbp",
    220050: "sbp",
    220180: "dbp",
    220051: "dbp",
    220181: "mbp",
    220052: "mbp",
    223761: "temperature",
    223762: "temperature",
}

value_ranges = {
    "heart_rate": (20, 300),
    "resp_rate": (1, 80),
    "spo2": (0, 100),
    "sbp": (20, 300),
    "dbp": (5, 200),
    "mbp": (10, 250),
    "temperature": (25, 45),
}

selected_itemids = set(chart_item_map)

In [4]:
d_items_df = pd.read_csv(RAW_DATA_DIR / "d_items.csv")

selected_items_df = d_items_df[d_items_df["itemid"].isin(selected_itemids)][
    ["itemid", "label", "unitname", "param_type"]
].copy()

selected_items_df["chart_feature"] = selected_items_df["itemid"].map(chart_item_map)
selected_items_df.sort_values(["chart_feature", "itemid"])

,itemid,label,unitname,param_type,chart_feature
7,220051,Arterial Blood Pressure diastolic,mmHg,Numeric,dbp
25,220180,Non Invasive Blood Pressure diastolic,mmHg,Numeric,dbp
2,220045,Heart Rate,bpm,Numeric,heart_rate
8,220052,Arterial Blood Pressure mean,mmHg,Numeric,mbp
26,220181,Non Invasive Blood Pressure mean,mmHg,Numeric,mbp
28,220210,Respiratory Rate,insp/min,Numeric,resp_rate
6,220050,Arterial Blood Pressure systolic,mmHg,Numeric,sbp
24,220179,Non Invasive Blood Pressure systolic,mmHg,Numeric,sbp
36,220277,O2 saturation pulseoxymetry,%,Numeric,spo2
337,223761,Temperature Fahrenheit,°F,Numeric,temperature


In [5]:
usecols = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "charttime",
    "itemid",
    "valuenum",
    "valueuom",
    "warning",
]

chunksize = 1_000_000
stay_ids = set(icustays_df["stay_id"])
clean_chunks = []

for chunk in pd.read_csv(
    RAW_DATA_DIR / "chartevents.csv.gz",
    usecols=usecols,
    chunksize=chunksize,
):
    chunk = chunk[
        chunk["stay_id"].isin(stay_ids)
        & chunk["itemid"].isin(selected_itemids)
        & chunk["valuenum"].notna()
    ].copy()

    if chunk.empty:
        continue

    chunk["charttime"] = pd.to_datetime(chunk["charttime"])
    chunk["chart_feature"] = chunk["itemid"].map(chart_item_map)

    temp_f_mask = chunk["itemid"].eq(223761)
    chunk.loc[temp_f_mask, "valuenum"] = (chunk.loc[temp_f_mask, "valuenum"] - 32) * 5 / 9
    chunk.loc[temp_f_mask, "valueuom"] = "degC"
    chunk.loc[chunk["itemid"].eq(223762), "valueuom"] = "degC"

    chunk = chunk.merge(
        icustays_df[["stay_id", "intime", "outtime"]],
        on="stay_id",
        how="inner"
    )

    chunk = chunk[
        chunk["charttime"].between(chunk["intime"], chunk["outtime"], inclusive="both")
    ].copy()

    chunk["hours_from_icu_admit"] = (
        (chunk["charttime"] - chunk["intime"])
        .dt.total_seconds()
        .div(3600)
    )

    valid_range_mask = pd.Series(True, index=chunk.index)
    for feature, (lower, upper) in value_ranges.items():
        feature_mask = chunk["chart_feature"].eq(feature)
        valid_range_mask &= ~feature_mask | chunk["valuenum"].between(lower, upper, inclusive="both")

    chunk = chunk[valid_range_mask].copy()

    clean_chunks.append(
        chunk[
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "charttime",
                "hours_from_icu_admit",
                "itemid",
                "chart_feature",
                "valuenum",
                "valueuom",
                "warning",
            ]
        ]
    )

if clean_chunks:
    chartevents_clean_df = pd.concat(clean_chunks, ignore_index=True)
else:
    chartevents_clean_df = pd.DataFrame(columns=[
        "subject_id",
        "hadm_id",
        "stay_id",
        "charttime",
        "hours_from_icu_admit",
        "itemid",
        "chart_feature",
        "valuenum",
        "valueuom",
        "warning",
    ])

print(chartevents_clean_df.shape)
print(chartevents_clean_df.isna().sum())
print(chartevents_clean_df["chart_feature"].value_counts())

(53537187, 10)
subject_id              0
hadm_id                 0
stay_id                 0
charttime               0
hours_from_icu_admit    0
itemid                  0
chart_feature           0
valuenum                0
valueuom                0
warning                 0
dtype: int64
chart_feature
heart_rate     8711106
resp_rate      8568042
spo2           8531645
sbp            8434305
dbp            8432319
mbp            8428002
temperature    2431768
Name: count, dtype: int64


In [6]:
chartevents_clean_df = chartevents_clean_df.sort_values(
    ["stay_id", "chart_feature", "charttime"]
).reset_index(drop=True)

print(chartevents_clean_df.duplicated().sum())
print(chartevents_clean_df[["hours_from_icu_admit", "valuenum"]].describe())

0
       hours_from_icu_admit      valuenum
count          5.353719e+07  5.353719e+07
mean           1.346766e+02  7.555337e+01
std            2.218143e+02  3.461680e+01
min            0.000000e+00  0.000000e+00
25%            2.060861e+01  5.200000e+01
50%            5.815750e+01  8.000000e+01
75%            1.582044e+02  9.800000e+01
max            5.432359e+03  3.000000e+02


In [7]:
chartevents_clean_df.to_parquet(
    PROCESSED_DATA_DIR / "chartevents_clean.parquet",
    index=False
)